In [ ]:
#!pip install xarray pystac_client kerchunk

In [ ]:
import json

import xarray as xr
from pystac_client import Client
from fsspec.implementations.reference import ReferenceNotReachable

catalog = Client.open("https://api.stac.ceda.ac.uk/")

In [ ]:
collection = catalog.get_collection("cmip6")

In [ ]:
renders = {}
for item in collection.get_items():
    variable = item.properties["cmip6:variable_id"]
    frequency = item.properties["cmip6:frequency"]
    if variable not in ["pr", "tas", "tasmax"]:
        continue
    if frequency not in ["mon", "monC"]:
        continue
        
    # only include priority-1 variables
    if item.properties["cmip6:table_id"] != "Amon":
        continue

    render_id = f"{variable}--{frequency}"
    print(render_id)
    if render_id in renders:
        continue

    vmin = None
    vmax = None
    try:
        with xr.open_dataset(item.assets["reference_file"].href, engine="kerchunk") as ds:
            da = ds[variable]

            vmin_start, vmax_start = map(float, da.isel(time=0).quantile((0.02, 0.98)))
            vmin_end, vmax_end = map(float, da.isel(time=-1).quantile((0.02, 0.98)))
            
            vmin = min(vmin_start, vmin_end)
            vmax = max(vmax_start, vmax_end)
    except FileNotFoundError as err:
        print(f"WARNING: Skipping due to FileNotFoundError: {err}")
        continue
    except ReferenceNotReachable as err:
        print(f"WARNING: Skipping due to ReferenceNotReachable: {err}")
        continue

    renders[render_id] = {
        "title": item.properties["cmip6:variable_long_name"].strip() + " (" + item.properties["cmip6:variable_units"] + ")",
        "assets": ["reference_file"],
        "variable": variable,
        "colormap_name": "viridis",
        "rescale": [[vmin, vmax]],
        "reference": True
    }    

In [ ]:
print(json.dumps(renders, indent=2))